<h1>Chapter 8 - Agentic RAG</h1>
<i>Building agents with MCP (Model Context Protocol) tools.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch08_agentic_rag/8.7_mcp_tools/mcp_tools.ipynb)

---

This notebook is for Chapter 8 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Building Agents with MCP Tools

The **Model Context Protocol (MCP)** is an open standard that allows AI agents to connect to external tools and services in a standardized way. In this notebook you will learn how to:

1. Connect to an MCP server (Playwright browser automation)
2. Explore available browser configurations
3. Connect to multiple MCP servers simultaneously
4. Build a web-browsing agent powered by MCP tools


## Install Required Packages

In [ ]:
pip install openai-agents

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
from google.colab import userdata
import os

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

## Example 1: Connect to a Playwright MCP Server

The Playwright MCP server exposes browser-automation capabilities as MCP tools. The snippet below connects to the server and lists the tools it provides.

In [ ]:
import asyncio
from agents.mcp import MCPServerStdio


async def connect_to_playwright():
    # Define connection parameters
    # "npx" runs Node.js packages, "@playwright/mcp@latest" specifies the package
    params = {"command": "npx", "args": ["@playwright/mcp@latest"]}

    # Create server connection with extended timeout
    async with MCPServerStdio(
        params=params, client_session_timeout_seconds=30  # Increased from default 5s
    ) as server:
        # List available tools from the server
        tools = await server.list_tools()
        print("Available tools:", tools)


# Run the async function
await connect_to_playwright()

## Example 2: Browser Configurations

The Playwright MCP server supports several browsers and launch options. The dictionary below shows the most common configurations.

In [ ]:
# Default browser (Chromium)
default_params = {"command": "npx", "args": ["@playwright/mcp@latest"]}

# Microsoft Edge
edge_params = {
    "command": "npx",
    "args": ["@playwright/mcp@latest", "--browser", "msedge"],
}

# Firefox
firefox_params = {
    "command": "npx",
    "args": ["@playwright/mcp@latest", "--browser", "firefox"],
}

# Chrome (if installed)
chrome_params = {
    "command": "npx",
    "args": ["@playwright/mcp@latest", "--browser", "chrome"],
}

# Headless mode (no GUI)
headless_params = {"command": "npx", "args": ["@playwright/mcp@latest", "--headless"]}

# Edge with custom viewport
edge_custom_params = {
    "command": "npx",
    "args": [
        "@playwright/mcp@latest",
        "--browser",
        "msedge",
        "--viewport-width",
        "1920",
        "--viewport-height",
        "1080",
    ],
}

print("Browser configurations defined successfully.")
print(f"Default params: {default_params}")
print(f"Headless params: {headless_params}")

## Example 3: Connecting to Multiple MCP Servers

Agents can use more than one MCP server at the same time. Here we connect to both a **filesystem** server and the **Playwright browser** server, and report how many tools each one provides.

In [ ]:
from agents.mcp import MCPServerStdio


async def connect_multiple_servers():
    # Filesystem server
    files_params = {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "."],
    }
    # Web browsing server
    browser_params = {"command": "npx", "args": ["@playwright/mcp@latest"]}

    async with MCPServerStdio(
        params=files_params, client_session_timeout_seconds=60
    ) as files:
        async with MCPServerStdio(
            params=browser_params, client_session_timeout_seconds=60
        ) as browser:

            file_tools = await files.list_tools()
            browser_tools = await browser.list_tools()

            print(f"File tools: {len(file_tools)}")
            print(f"Browser tools: {len(browser_tools)}")


await connect_multiple_servers()

## Example 4: Create a Web Agent Powered by Playwright

Combining the filesystem and browser MCP servers, we can build an agent that browses the web **and** saves results to local files — all without writing any custom tool code.

In [ ]:
from agents.mcp import MCPServerStdio
from agents import Agent, Runner


async def create_cookie_research_agent():
    # MCP server configurations
    files_params = {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "."],
    }
    browser_params = {"command": "npx", "args": ["@playwright/mcp@latest"]}

    # Agent instructions
    instructions = "You can browse websites and save information to files."

    async with MCPServerStdio(
        params=files_params, client_session_timeout_seconds=60
    ) as files:
        async with MCPServerStdio(
            params=browser_params, client_session_timeout_seconds=60
        ) as browser:

            # Create agent
            agent = Agent(
                name="research_agent",
                instructions=instructions,
                model="gpt-4o-mini",
                mcp_servers=[files, browser],
            )

            # Run task
            result = await Runner.run(
                agent, "Find a chocolate chip cookie recipe and save it to recipe.md"
            )

            print(result.final_output)


await create_cookie_research_agent()